# AI014 - Metric Execution Demo

This notebook tests the metric execution code for AI014. It uses existing baseline prediction output when available. If the file is not available, it uses a small fallback sample based on the AI009 validation output.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

# Find evaluation root whether the notebook is run from repo root or from the notebooks folder.
cwd = Path.cwd()
if cwd.name == "notebooks":
    evaluation_root = cwd.parent
elif (cwd / "src" / "metrics").exists():
    evaluation_root = cwd
elif (cwd / "ai-ml" / "evaluation").exists():
    evaluation_root = cwd / "ai-ml" / "evaluation"
else:
    evaluation_root = cwd.parent

src_path = evaluation_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metrics.metric_utils import evaluate_prediction_dataframe, save_metrics_json

evaluation_root

In [ ]:
# Try to load AI009 baseline predictions. If not found, use a small fallback sample.
candidate_paths = [
    evaluation_root.parent / "ai009_baseline" / "outputs" / "predictions_val.csv",
    evaluation_root / "predictions_val.csv",
    Path.cwd() / "predictions_val.csv",
]

input_path = next((path for path in candidate_paths if path.exists()), None)

if input_path:
    df = pd.read_csv(input_path)
    source = str(input_path)
else:
    df = pd.DataFrame({
        "temperature": [29, 47],
        "humidity": [62, 27],
        "actual_label": [0, 1],
        "predicted_label": [0, 1],
        "predicted_probability": [0.23794634949659663, 0.9982779224071671],
    })
    source = "fallback sample based on AI009 validation output"

print(f"Input source: {source}")
df

In [ ]:
result = evaluate_prediction_dataframe(
    df,
    actual_col="actual_label",
    predicted_col="predicted_label",
    probability_col="predicted_probability",
    metadata={"source": source},
)

print(json.dumps(result["metrics"], indent=2))

In [ ]:
output_path = evaluation_root / "outputs" / "metrics" / "evaluation_metrics.json"
save_metrics_json(result, output_path)
print(f"Saved metric output to: {output_path}")